# Student Performance Prediction in Bangladesh
This notebook demonstrates a complete machine learning pipeline to predict HSC results based on student demographics, family background, and previous academic performance (SSC results).

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## 1. Data Loading and Initial Exploration
In this section, we load the dataset and perform a quick check of the data structure, including shapes, samples, and summary statistics to understand the features we are working with.

In [2]:
# Load the student performance dataset from a CSV file
df = pd.read_csv("bangladesh_student_performance_updated.csv")

In [3]:
df.head(10)

,date,gender,age,address,famsize,Pstatus,M_Edu,F_Edu,M_Job,F_Job,relationship,smoker,tuition_fee,time_friends,ssc_result,hsc_result
0,29/04/2018,M,18.0,Rural,GT3,Together,3,2,At_home,Farmer,No,No,71672.0,4,4.22,3.72
1,29/04/2018,F,19.0,Rural,LE3,Apart,0,4,Other,Health,Yes,No,26085.0,5,3.47,2.62
2,29/04/2018,F,19.0,Rural,GT3,Together,0,3,Teacher,Services,No,No,40891.0,3,3.32,2.56
3,29/04/2018,F,19.0,Rural,LE3,Apart,2,3,At_home,Business,No,No,50600.0,2,4.57,4.17
4,29/04/2018,M,17.0,Rural,GT3,Together,1,1,At_home,Farmer,No,No,62458.0,2,4.50,3.94
5,29/04/2018,M,18.0,Rural,GT3,Together,2,4,At_home,Farmer,No,Yes,43645.0,2,4.09,3.65
6,29/04/2018,M,18.0,Urban,GT3,Together,1,0,Teacher,Teacher,No,No,111578.0,1,3.46,2.99
7,29/04/2018,F,18.0,Rural,GT3,Together,2,1,Services,Teacher,Yes,No,76596.0,3,4.51,3.92
8,29/04/2018,F,17.0,Rural,GT3,Together,2,3,Other,Teacher,Yes,Yes,39874.0,3,3.26,2.59
9,29/04/2018,F,17.0,Rural,LE3,Together,2,2,Health,Health,No,No,48767.0,3,4.72,4.25


In [4]:
df.shape

(2018, 16)

In [5]:
df.sample(10)

,date,gender,age,address,famsize,Pstatus,M_Edu,F_Edu,M_Job,F_Job,relationship,smoker,tuition_fee,time_friends,ssc_result,hsc_result
639,29/04/2018,M,17.0,Rural,GT3,Together,3,2,Other,Services,No,No,58755.0,3,4.01,3.54
1541,29/04/2018,M,17.0,Urban,GT3,Apart,0,4,Teacher,Teacher,No,No,73843.0,1,5.00,4.66
1346,29/04/2018,M,19.0,Rural,GT3,Together,1,1,Services,Teacher,No,No,35690.0,5,3.52,2.97
200,29/04/2018,F,18.0,Rural,LE3,Together,3,4,Health,Farmer,No,No,62660.0,2,4.74,4.59
650,29/04/2018,M,19.0,Rural,GT3,Together,3,3,Other,Business,No,No,43159.0,4,2.51,2.12
1328,29/04/2018,F,19.0,Rural,LE3,Together,4,0,Other,Teacher,No,No,61832.0,5,4.88,4.39
1780,29/04/2018,M,19.0,Rural,LE3,Together,2,3,Health,Business,No,No,68724.0,4,3.35,2.61
1798,29/04/2018,M,17.0,Urban,LE3,Together,2,0,Teacher,Business,No,No,103667.0,5,4.73,4.12
618,29/04/2018,F,17.0,Urban,GT3,Apart,2,4,Health,Farmer,No,No,119764.0,4,5.00,4.45
1692,29/04/2018,M,18.0,Urban,GT3,Together,3,2,Other,Farmer,No,No,102487.0,4,4.20,3.75


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2018 entries, 0 to 2017
Data columns (total 16 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   date          2018 non-null   object 
 1   gender        2018 non-null   object 
 2   age           1983 non-null   float64
 3   address       1983 non-null   object 
 4   famsize       2018 non-null   object 
 5   Pstatus       2018 non-null   object 
 6   M_Edu         2018 non-null   int64  
 7   F_Edu         2018 non-null   int64  
 8   M_Job         1983 non-null   object 
 9   F_Job         2018 non-null   object 
 10  relationship  2018 non-null   object 
 11  smoker        2018 non-null   object 
 12  tuition_fee   1983 non-null   float64
 13  time_friends  2018 non-null   int64  
 14  ssc_result    2018 non-null   float64
 15  hsc_result    2018 non-null   float64
dtypes: float64(4), int64(3), object(9)
memory usage: 252.4+ KB


In [10]:
df['time_friends'].unique()
df['M_Edu'].unique()
df['famsize'].unique()

array(['GT3', 'LE3'], dtype=object)

In [11]:
df.isnull().sum()

date             0
gender           0
age             35
address         35
famsize          0
Pstatus          0
M_Edu            0
F_Edu            0
M_Job           35
F_Job            0
relationship     0
smoker           0
tuition_fee     35
time_friends     0
ssc_result       0
hsc_result       0
dtype: int64

In [12]:
df.describe()

,age,M_Edu,F_Edu,tuition_fee,time_friends,ssc_result,hsc_result
count,1983.000000,2018.000000,2018.000000,1983.000000,2018.00000,2018.000000,2018.000000
mean,17.980837,1.871160,2.174430,73045.274332,3.05996,3.788087,3.199177
std,0.825695,1.194206,1.252979,24102.016020,1.43919,0.622376,0.604526
min,17.000000,0.000000,0.000000,25102.000000,1.00000,2.000000,2.000000
25%,17.000000,1.000000,1.000000,53639.500000,2.00000,3.360000,2.780000
50%,18.000000,2.000000,2.000000,71343.000000,3.00000,3.770000,3.160000
75%,19.000000,3.000000,3.000000,91185.000000,4.00000,4.230000,3.580000
max,19.000000,4.000000,4.000000,129168.000000,5.00000,5.000000,5.000000
